# 04a - Estimasi & Inferensi
Materi Modul 4: estimasi, parametrik vs non-parametrik, distribusi umum, statistik frequentist vs Bayesian.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)
peng = pd.read_csv('../dataset/penguins.csv')
dia = pd.read_csv('../dataset/diabetes.csv')

## 1. Distribusi data & uji normalitas

In [2]:
x = peng['body_mass_g']
mu, sd = x.mean(), x.std()
grid = np.linspace(x.min(), x.max(), 200)
plt.hist(x, bins=20, density=True, alpha=.6)
plt.plot(grid, stats.norm.pdf(grid, mu, sd), 'r')
plt.title('Body mass vs kurva normal'); plt.show()
print('Shapiro-Wilk p-value:', round(stats.shapiro(x).pvalue, 4))

Shapiro-Wilk p-value: 0.0


## 2. Estimasi rata-rata: parametrik vs non-parametrik (bootstrap)

In [3]:
n = len(x)
ci_t = stats.t.interval(0.95, df=n-1, loc=mu, scale=stats.sem(x))
boot = [rng.choice(x, n, replace=True).mean() for _ in range(5000)]
ci_boot = np.percentile(boot, [2.5, 97.5])
print('Rata-rata        :', round(mu, 3))
print('CI 95% t-interval:', np.round(ci_t, 3))
print('CI 95% bootstrap :', np.round(ci_boot, 3))

Rata-rata        : 4207.057
CI 95% t-interval: [4120.256 4293.858]
CI 95% bootstrap : [4119.968 4293.255]


## 3. Frequentist vs Bayesian
Mengestimasi proporsi pasien diabetes (`target=1`) pada data Pima Indians Diabetes.

In [4]:
k, n = int(dia['target'].sum()), len(dia)
p_hat = k / n
# Frequentist: Wald 95% CI
se = np.sqrt(p_hat*(1-p_hat)/n)
print('Frequentist  p =', round(p_hat,4), '| CI 95% =', np.round([p_hat-1.96*se, p_hat+1.96*se], 4))
# Bayesian: prior Beta(1,1) -> posterior Beta(1+k, 1+n-k)
post = stats.beta(1+k, 1+n-k)
print('Bayesian     mean =', round(post.mean(),4), '| Credible interval 95% =', np.round(post.interval(0.95), 4))

Frequentist  p = 0.349 | CI 95% = [0.3152 0.3827]
Bayesian     mean = 0.3494 | Credible interval 95% = [0.3161 0.3834]


In [5]:
xs = np.linspace(0.29, 0.41, 300)
plt.plot(xs, post.pdf(xs)); plt.axvline(p_hat, color='r', ls='--')
plt.title('Distribusi posterior proporsi pasien diabetes'); plt.show()

## Catatan
- (beda interpretasi confidence interval vs credible interval; kapan non-parametrik dipilih)